In [ ]:

"""
Phi-4-mini-instruct QLoRA Fine-tuning  Version 3 (Improved)
===============================================================
Key improvements over v2:
  1. MAX_LABEL_COUNT = 1500  (was 5000) → stops "cited" dominating
  2. WeightedSFTTrainer with inverse-frequency class weights
  3. num_train_epochs = 2    (was 1)
  4. learning_rate = 1e-4    (was 2e-4) → safer with weighted loss
  5. Richer system prompt with per-label descriptions

Dataset : darklord1611/legal_citations  (HuggingFace Hub)
Model   : microsoft/Phi-4-mini-instruct
Save to : /content/drive/MyDrive/phi4_thesis_HuggingFace_v3
"""


'\nPhi-4-mini-instruct QLoRA Fine-tuning  —  Version 3 (Improved)\n===============================================================\nKey improvements over v2:\n  1. MAX_LABEL_COUNT = 1500  (was 5000) → stops "cited" dominating\n  2. WeightedSFTTrainer with inverse-frequency class weights\n  3. num_train_epochs = 2    (was 1)\n  4. learning_rate = 1e-4    (was 2e-4) → safer with weighted loss\n  5. Richer system prompt with per-label descriptions\n\nDataset : darklord1611/legal_citations  (HuggingFace Hub)\nModel   : microsoft/Phi-4-mini-instruct\nSave to : /content/drive/MyDrive/phi4_thesis_HuggingFace_v3\n'

In [ ]:

#   Install / upgrade all dependencies
#   Run this cell, then RESTART THE RUNTIME before running Cell 2+

import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "torch", "torchvision", "torchaudio",
                "transformers", "peft", "trl", "accelerate", "bitsandbytes"],
               capture_output=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu124"],
               check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.48.1",
                "accelerate==1.3.0",
                "peft==0.14.0",
                "trl==0.15.2",
                "bitsandbytes>=0.43.0",
                "datasets>=2.18.0",
                "scikit-learn",
                "pandas",
                "numpy",
                "sentencepiece",
                "protobuf"],
               check=True)

print(" All packages installed — RESTART THE RUNTIME NOW, then run Cell 2+")

✅  All packages installed — RESTART THE RUNTIME NOW, then run Cell 2+


In [ ]:

# Mount Google Drive & HuggingFace login
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import login
HF_TOKEN = "hf_***********"   
login(HF_TOKEN)
print("Drive mounted & HF login complete.")

Mounted at /content/drive
✅  Drive mounted & HF login complete.


In [ ]:
# Imports & reproducibility seeds

import os, json, random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✅  Imports done.")
print("    GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch to GPU runtime!")

✅  Imports done.
    GPU: NVIDIA A100-SXM4-80GB


In [ ]:
# Project paths

BASE_MODEL_ID   = "microsoft/Phi-4-mini-instruct"
HF_DATASET_ID   = "darklord1611/legal_citations"

#  New v3 directory so it doesn't overwrite your v2 run
PROJECT_DIR     = "/content/drive/MyDrive/phi4_thesis_HuggingFace_v3"
SPLITS_DIR      = os.path.join(PROJECT_DIR, "splits")
QLORA_DIR       = os.path.join(PROJECT_DIR, "qlora_model")
METRICS_DIR     = os.path.join(PROJECT_DIR, "metrics")
PREDICTIONS_DIR = os.path.join(PROJECT_DIR, "predictions")

for folder in [PROJECT_DIR, SPLITS_DIR, QLORA_DIR, METRICS_DIR, PREDICTIONS_DIR]:
    os.makedirs(folder, exist_ok=True)

print("✅  Directories ready under:", PROJECT_DIR)

✅  Directories ready under: /content/drive/MyDrive/phi4_thesis_HuggingFace_v3


In [ ]:
# Load dataset from HuggingFace

print(f"Loading {HF_DATASET_ID} …")
raw = load_dataset(HF_DATASET_ID)

print("Splits    :", list(raw.keys()))
print("Train rows:", len(raw["train"]))
print("Test  rows:", len(raw["test"]))
print("Columns   :", raw["train"].column_names)
print("\nSample row:")
print(raw["train"][0])


Loading darklord1611/legal_citations …


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/58.7M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21237 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3748 [00:00<?, ? examples/s]

Splits    : ['train', 'test']
Train rows: 21237
Test  rows: 3748
Columns   : ['Unnamed: 0', 'case_id', 'case_outcome', 'case_title', 'case_text']

Sample row:
{'Unnamed: 0': 21382, 'case_id': 'Case21569', 'case_outcome': 'distinguished', 'case_title': 'Housing Guarantee Fund Ltd v Yusef [1991] 2 VR 17', 'case_text': "The second issue to which I referred, can best be considered by reference to Housing Guarantee Fund Ltd v Yusef [1991] 2 VR 17 (' HGFL v Yusef ')."}


In [ ]:
#  Convert to DataFrames & normalise

TEXT_COL  = "case_text"
LABEL_COL = "case_outcome"

train_df_raw = raw["train"].to_pandas()
test_df_raw  = raw["test"].to_pandas()

# Normalise column names defensively
for df in [train_df_raw, test_df_raw]:
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Validate required columns exist
for col in [TEXT_COL, LABEL_COL]:
    assert col in train_df_raw.columns, \
        f"Column '{col}' not found. Available: {train_df_raw.columns.tolist()}"

train_df_raw = train_df_raw[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)
test_df_raw  = test_df_raw[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)

for df in [train_df_raw, test_df_raw]:
    df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()

print("Raw train shape:", train_df_raw.shape)
print("Raw test  shape:", test_df_raw.shape)
print("\nRaw label distribution (train):")
print(train_df_raw[LABEL_COL].value_counts().to_string())

Raw train shape: (21083, 2)
Raw test  shape: (3726, 2)

Raw label distribution (train):
case_outcome
cited            10296
referred to       3727
applied           2051
followed          1923
considered        1449
discussed          860
distinguished      505
approved            97
related             91
affirmed            84


In [ ]:
#   Class-imbalance correction
#   MIN_LABEL_COUNT = 100   → drop rare classes (not enough signal)
#   MAX_LABEL_COUNT = 1500  → cap dominant classes (stops "cited" bias)
#
#   Thesis justification:
#   Undersampling is applied to reduce the maximum class imbalance ratio
#   from ~40:1 to under 5:1, following He & Garcia (2009). A cap of 1500
#   retains substantial signal per class while preventing the model from
#   learning a majority-class bias. The test set is NOT rebalanced so
#   evaluation reflects real-world distribution.

MIN_LABEL_COUNT = 100
MAX_LABEL_COUNT = 1500

label_counts = train_df_raw[LABEL_COL].value_counts()
print("── Before balancing (train label counts) ──")
print(label_counts.to_string())

# 1) Remove labels below minimum threshold
valid_labels = label_counts[label_counts >= MIN_LABEL_COUNT].index.tolist()
removed      = label_counts[label_counts <  MIN_LABEL_COUNT].index.tolist()
print(f"\nLabels kept   (≥{MIN_LABEL_COUNT} rows): {valid_labels}")
print(f"Labels removed (<{MIN_LABEL_COUNT} rows): {removed}")

train_df_filtered = train_df_raw[train_df_raw[LABEL_COL].isin(valid_labels)].copy()

# 2) Cap oversized labels with random undersampling
def cap_label_group(group, max_count, seed):
    if len(group) > max_count:
        return group.sample(n=max_count, random_state=seed)
    return group

train_df_balanced = (
    train_df_filtered
    .groupby(LABEL_COL, group_keys=False)
    .apply(lambda g: cap_label_group(g, MAX_LABEL_COUNT, SEED))
    .reset_index(drop=True)
)

print("\n── After balancing (train label counts) ──")
print(train_df_balanced[LABEL_COL].value_counts().to_string())
print("\nBalanced train shape:", train_df_balanced.shape)

LABELS       = sorted(train_df_balanced[LABEL_COL].unique().tolist())
LABELS_LOWER = [l.lower().strip() for l in LABELS]
print("\nFinal label set:", LABELS)

── Before balancing (train label counts) ──
case_outcome
cited            10296
referred to       3727
applied           2051
followed          1923
considered        1449
discussed          860
distinguished      505
approved            97
related             91
affirmed            84

Labels kept   (≥100 rows): ['cited', 'referred to', 'applied', 'followed', 'considered', 'discussed', 'distinguished']
Labels removed (<100 rows): ['approved', 'related', 'affirmed']

── After balancing (train label counts) ──
case_outcome
applied          1500
cited            1500
referred to      1500
followed         1500
considered       1449
discussed         860
distinguished     505

Balanced train shape: (8814, 2)

Final label set: ['applied', 'cited', 'considered', 'discussed', 'distinguished', 'followed', 'referred to']


/tmp/ipykernel_824/4022575424.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: cap_label_group(g, MAX_LABEL_COUNT, SEED))


In [ ]:
#   Split HF test → validation + test (50 / 50, stratified)
#   Test set is NOT rebalanced — preserves natural distribution for
#   realistic evaluation (important for thesis credibility)
test_df_filtered = test_df_raw[test_df_raw[LABEL_COL].isin(LABELS)].reset_index(drop=True)

label_counts_test = test_df_filtered[LABEL_COL].value_counts()
can_stratify      = label_counts_test.min() >= 2

val_df, test_df = train_test_split(
    test_df_filtered,
    test_size=0.50,
    random_state=SEED,
    stratify=test_df_filtered[LABEL_COL] if can_stratify else None,
)
val_df  = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape      :", train_df_balanced.shape)
print("Validation shape :", val_df.shape)
print("Test shape       :", test_df.shape)
print("\nValidation label distribution:")
print(val_df[LABEL_COL].value_counts().to_string())
print("\nTest label distribution (natural — not rebalanced):")
print(test_df[LABEL_COL].value_counts().to_string())

# Save splits
train_df_balanced.to_csv(os.path.join(SPLITS_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(SPLITS_DIR,  "val.csv"),             index=False)
test_df.to_csv(os.path.join(SPLITS_DIR, "test.csv"),            index=False)
with open(os.path.join(SPLITS_DIR, "labels.json"), "w") as f:
    json.dump(LABELS, f, indent=2)

print("\n✅  Splits saved to:", SPLITS_DIR)

Train shape      : (8814, 2)
Validation shape : (1836, 2)
Test shape       : (1836, 2)

Validation label distribution:
case_outcome
cited            907
referred to      318
applied          193
followed         165
considered       125
discussed         79
distinguished     49

Test label distribution (natural — not rebalanced):
case_outcome
cited            907
referred to      318
applied          194
followed         164
considered       125
discussed         79
distinguished     49

✅  Splits saved to: /content/drive/MyDrive/phi4_thesis_HuggingFace_v3/splits


In [ ]:
#   Compute inverse-frequency class weights
#
#   Weight formula: w_c = N / (K × n_c)
#     N  = total training samples
#     K  = number of classes
#     n_c = samples in class c
#   Rare classes get higher weight → model penalised more for missing them.
# ===========================================================================
label_freq    = Counter(train_df_balanced[LABEL_COL].tolist())
total_samples = sum(label_freq.values())
n_classes     = len(LABELS)

class_weight_values = []
print("── Class weights (higher = rarer class, penalised more) ──")
for label in LABELS:
    freq = label_freq.get(label, 1)
    w    = total_samples / (n_classes * freq)
    class_weight_values.append(w)
    print(f"  {label:<20} : freq={freq:>5}  weight={w:.4f}")

class_weights = torch.tensor(class_weight_values, dtype=torch.float32)
print("\n✅  Class weights computed.")


── Class weights (higher = rarer class, penalised more) ──
  applied              : freq= 1500  weight=0.8394
  cited                : freq= 1500  weight=0.8394
  considered           : freq= 1449  weight=0.8690
  discussed            : freq=  860  weight=1.4641
  distinguished        : freq=  505  weight=2.4934
  followed             : freq= 1500  weight=0.8394
  referred to          : freq= 1500  weight=0.8394

✅  Class weights computed.


In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
)
tokenizer.model_max_length = 2048
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"   # right-pad during training

print("✅  Tokenizer loaded.")
print("    Vocab size:", tokenizer.vocab_size)
print("    Pad token :", repr(tokenizer.pad_token))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

✅  Tokenizer loaded.
    Vocab size: 200019
    Pad token : '<|endoftext|>'


In [ ]:
#   Build prompt / chat-template formatting
#   Improvement over v2: richer system prompt with per-label descriptions
#   gives the model clear semantic anchors for each class.

SYSTEM_MSG = (
    "You are a legal assistant specialising in case-outcome classification. "
    "Analyse the provided case text and return exactly one label from the "
    "allowed list — nothing else.\n\n"
    "Label definitions:\n"
    "- cited: The case was cited as authority without further treatment.\n"
    "- applied: The legal principle from the case was directly applied to the facts.\n"
    "- followed: The decision was followed as binding or persuasive precedent.\n"
    "- considered: The case was examined and taken into account but not necessarily followed.\n"
    "- discussed: The case was discussed in detail, analysing its reasoning or implications.\n"
    "- distinguished: The case was differentiated on its facts or legal principles.\n"
    "- referred to: The case was briefly mentioned as a passing reference.\n"
)

LABEL_STR = ", ".join(LABELS)

def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {
            "role": "user",
            "content": (
                f"Case Text:\n{example[TEXT_COL]}\n\n"
                f"Allowed labels: {LABEL_STR}.\n"
                "What is the outcome of this case? Return only the label."
            ),
        },
        {
            "role": "assistant",
            "content": str(example[LABEL_COL]).strip().lower(),
        },
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

In [ ]:
# Create HuggingFace Dataset objects & apply template
raw_ds = DatasetDict({
    "train":      Dataset.from_pandas(train_df_balanced),
    "validation": Dataset.from_pandas(val_df),
})

processed_train = raw_ds["train"].map(
    format_example,
    remove_columns=raw_ds["train"].column_names,
    desc="Formatting train",
)
processed_val = raw_ds["validation"].map(
    format_example,
    remove_columns=raw_ds["validation"].column_names,
    desc="Formatting validation",
)

print("Processed train size :", len(processed_train))
print("Processed val size   :", len(processed_val))
print("\n── Sample formatted text (first 800 chars) ──")
print(processed_train[0]["text"][:800])

Formatting train:   0%|          | 0/8814 [00:00<?, ? examples/s]

Formatting validation:   0%|          | 0/1836 [00:00<?, ? examples/s]

Processed train size : 8814
Processed val size   : 1836

── Sample formatted text (first 800 chars) ──
<|system|>You are a legal assistant specialising in case-outcome classification. Analyse the provided case text and return exactly one label from the allowed list — nothing else.

Label definitions:
- cited: The case was cited as authority without further treatment.
- applied: The legal principle from the case was directly applied to the facts.
- followed: The decision was followed as binding or persuasive precedent.
- considered: The case was examined and taken into account but not necessarily followed.
- discussed: The case was discussed in detail, analysing its reasoning or implications.
- distinguished: The case was differentiated on its facts or legal principles.
- referred to: The case was briefly mentioned as a passing reference.
<|end|><|user|>Case Text:
Finance Limited v Commissio


In [ ]:
# 4-bit quantisation config (QLoRA)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
print("✅  BitsAndBytes config ready.")

✅  BitsAndBytes config ready.


In [ ]:

# Load model in 4-bit & prepare for k-bit training

print(f"Loading {BASE_MODEL_ID} in 4-bit …  (may take 1–2 min)")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    use_cache=False,
    attn_implementation="eager",
)
model = prepare_model_for_kbit_training(model)

print("✅  Model loaded.")
print(f"    Trainable params (before LoRA): "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Loading microsoft/Phi-4-mini-instruct in 4-bit …  (may take 1–2 min)


config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

✅  Model loaded.
    Trainable params (before LoRA): 0


In [ ]:
# LoRA / PEFT config
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
print(f"✅  LoRA config ready.  r={peft_config.r}  alpha={peft_config.lora_alpha}")

✅  LoRA config ready.  r=32  alpha=64


In [ ]:
# WeightedSFTTrainer  (FIXED — uses model vocab size not tokenizer)
# Get the true vocab size from the model config (matches logits dimension)
true_vocab_size = model.config.vocab_size
print(f"Model vocab size (from config): {true_vocab_size}")

# Build a vocab-sized weight tensor — all tokens default to 1.0
# Only the 7 label token positions get inverse-frequency weights
label_token_weights = torch.ones(true_vocab_size, dtype=torch.float32)

for label, weight in zip(LABELS, class_weight_values):
    token_ids = tokenizer.encode(label, add_special_tokens=False)
    if token_ids:
        tid = token_ids[0]
        if tid < true_vocab_size:
            label_token_weights[tid] = weight
            print(f"  {label:<20} → token_id={tid}  weight={weight:.4f}")

print(f"\n✅  Weight tensor shape: {label_token_weights.shape}")


class WeightedSFTTrainer(SFTTrainer):
    """SFTTrainer with vocab-sized inverse-frequency weighted cross-entropy."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fn = nn.CrossEntropyLoss(
            weight=label_token_weights.to(shift_logits.device),
            ignore_index=-100,
        )
        loss = loss_fn(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss

print("✅  WeightedSFTTrainer defined.")

Model vocab size (from config): 200064
  applied              → token_id=403  weight=0.8394
  cited                → token_id=66  weight=0.8394
  considered           → token_id=59464  weight=0.8690
  discussed            → token_id=4220  weight=1.4641
  distinguished        → token_id=24126  weight=2.4934
  followed             → token_id=34865  weight=0.8394
  referred to          → token_id=264  weight=0.8394

✅  Weight tensor shape: torch.Size([200064])
✅  WeightedSFTTrainer defined.


In [ ]:
#    SFT training config
#
#   Key settings vs v2:
#     • num_train_epochs = 2     (was 1)
#     • learning_rate   = 1e-4   (was 2e-4; safer with weighted loss)
#     • eval/save every 300 steps (good visibility across 2 epochs)
# ===========================================================================
_PER_DEVICE_BATCH = 2
_GRAD_ACCUM       = 4
_EFFECTIVE_BATCH  = _PER_DEVICE_BATCH * _GRAD_ACCUM   # = 8

sft_config = SFTConfig(
    #  Output
    output_dir=QLORA_DIR,

    #  Precision 
    bf16=True,

    #  Epochs & steps
    num_train_epochs=2,   # 2 full epochs — feasible on A100 with 1500-cap data
    max_steps=-1,         # honour num_train_epochs (no step cap)

    #  Batch & accumulation 
    per_device_train_batch_size=_PER_DEVICE_BATCH,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=_GRAD_ACCUM,

    #  Learning rate 
    learning_rate=1e-4,         # reduced from 2e-4; safer with weighted loss
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    #  Memory 
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",

    #  Logging 
    logging_strategy="steps",
    logging_steps=20,
    report_to="none",

    #  Evaluation 
    do_eval=True,
    eval_strategy="steps",
    eval_steps=300,

    #  Checkpointing 
    save_strategy="steps",
    save_steps=300,
    save_total_limit=2,

    #  Dataset 
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    remove_unused_columns=True,

    #  Best-model restoration
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

print("✅  SFTConfig ready.")
print(f"   Epochs      : {sft_config.num_train_epochs}")
print(f"   max_steps   : {sft_config.max_steps}  (-1 = full epochs)")
print(f"   LR          : {sft_config.learning_rate}")
print(f"   Eff. batch  : {_PER_DEVICE_BATCH} × {_GRAD_ACCUM} = {_EFFECTIVE_BATCH}")
print(f"   Eval every  : {sft_config.eval_steps} steps")

✅  SFTConfig ready.
   Epochs      : 2
   max_steps   : -1  (-1 = full epochs)
   LR          : 0.0001
   Eff. batch  : 2 × 4 = 8
   Eval every  : 300 steps


In [ ]:
#  Build trainer & start training

trainer = WeightedSFTTrainer(
    model=model,
    args=sft_config,
    peft_config=peft_config,
    train_dataset=processed_train,
    eval_dataset=processed_val,
    processing_class=tokenizer,
)

print(f"Trainable parameters after LoRA: "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

print("\n🚀  Starting training …")
train_result   = trainer.train()
train_metrics  = train_result.metrics

print("\n── Training metrics ──")
print(json.dumps(train_metrics, indent=2))

Converting train dataset to ChatML:   0%|          | 0/8814 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/8814 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8814 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8814 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/1836 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/1836 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1836 [00:00<?, ? examples/s]

Trainable parameters after LoRA: 46,137,344

🚀  Starting training …


Step,Training Loss,Validation Loss
300,6.263900,1.491323
600,6.138200,1.438070
900,5.858700,1.403000
1200,5.119100,1.377157
1500,5.014800,1.361392
1800,5.268600,1.347876
2100,5.389300,1.344200



── Training metrics ──
{
  "train_runtime": 8792.3583,
  "train_samples_per_second": 2.005,
  "train_steps_per_second": 0.25,
  "total_flos": 3.257289933025075e+17,
  "train_loss": 5.673725965778791
}


In [ ]:
# Save model, tokenizer, labels & metrics

trainer.save_model(QLORA_DIR)
tokenizer.save_pretrained(QLORA_DIR)

# Save label list alongside adapter for self-contained evaluation
with open(os.path.join(QLORA_DIR,  "labels.json"), "w") as f:
    json.dump(LABELS, f, indent=2)
with open(os.path.join(SPLITS_DIR, "labels.json"), "w") as f:
    json.dump(LABELS, f, indent=2)

with open(os.path.join(METRICS_DIR, "train_metrics.json"), "w") as f:
    json.dump(train_metrics, f, indent=2)

print("✅  Model adapter saved to:", QLORA_DIR)
print("   Files:", os.listdir(QLORA_DIR))


✅  Model adapter saved to: /content/drive/MyDrive/phi4_thesis_HuggingFace_v3/qlora_model
   Files: ['checkpoint-2100', 'checkpoint-2202', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'training_args.bin', 'labels.json']


In [ ]:
# Final validation-loss evaluation & save

eval_metrics = trainer.evaluate()
eval_metrics["eval_samples"] = len(processed_val)

print("\n Validation metrics ")
print(json.dumps(eval_metrics, indent=2))

with open(os.path.join(METRICS_DIR, "eval_loss_metrics.json"), "w") as f:
    json.dump(eval_metrics, f, indent=2)

print("\n✅  Fine-tuning v3 complete.")
print("    QLoRA adapter :", QLORA_DIR)
print("    Splits        :", SPLITS_DIR)
print("    Metrics       :", METRICS_DIR)
print("\nNext step: run phi4_legal_evaluation_v3.py on the test split.")


── Validation metrics ──
{
  "eval_loss": 1.344199776649475,
  "eval_runtime": 197.9673,
  "eval_samples_per_second": 9.274,
  "eval_steps_per_second": 4.637,
  "eval_samples": 1836
}

✅  Fine-tuning v3 complete.
    QLoRA adapter : /content/drive/MyDrive/phi4_thesis_HuggingFace_v3/qlora_model
    Splits        : /content/drive/MyDrive/phi4_thesis_HuggingFace_v3/splits
    Metrics       : /content/drive/MyDrive/phi4_thesis_HuggingFace_v3/metrics

Next step: run phi4_legal_evaluation_v3.py on the test split.
